# Task1 Evaluation code

### Final score = 0.7 × Patient-level cavity detection accuracy + 0.3 × DSC

In [ ]:
import os
import glob
import numpy as np
import SimpleITK as sitk
from tqdm.notebook import tqdm

DIR_GT_MASK = "/workspace/treat_mmtb/nas125/IDs/hyekyojeong/model/nnUNetv2/nnUNet_data/nnUNet_raw/Dataset001_Task1/labelsTr"
DIR_PRED_MASK = "/workspace/treat_mmtb/nas125/IDs/hyekyojeong/model/nnUNetv2/nnUNet_data/nnUNet_results/Dataset001_Task1/nnUNetTrainer__nnUNetPlans__2d/fold_0/validation"

def load_binary_mask(path):
    mask = sitk.GetArrayFromImage(sitk.ReadImage(path))
    mask = np.squeeze(mask)
    return (mask > 0).astype(np.uint8)

def dice_score(pred, gt):
    pred_sum = pred.sum()
    gt_sum = gt.sum()

    if pred_sum == 0 and gt_sum == 0:
        return np.nan
    if pred_sum == 0 and gt_sum > 0:
        return 0.0
    if pred_sum > 0 and gt_sum == 0:
        return 0.0

    intersection = np.logical_and(pred, gt).sum()
    return 2.0 * intersection / (pred_sum + gt_sum)

def evaluate_detection_and_segmentation(gt_dir, pred_dir):
    gt_paths = sorted(glob.glob(os.path.join(gt_dir, "*.nii.gz")))

    case_results = []

    for gt_path in tqdm(gt_paths):
        case_id = os.path.basename(gt_path).replace(".nii.gz", "")
        pred_path = os.path.join(pred_dir, f"{case_id}.nii.gz")

        if not os.path.exists(pred_path):
            raise FileNotFoundError(f"Missing prediction for {case_id}: {pred_path}")

        gt = load_binary_mask(gt_path)
        pred = load_binary_mask(pred_path)

        gt_positive = gt.sum() > 0
        pred_positive = pred.sum() > 0

        acc = float(gt_positive == pred_positive)
        dsc = dice_score(pred, gt)

        case_results.append({
            "case_id": case_id,
            "gt_positive": bool(gt_positive),
            "pred_positive": bool(pred_positive),
            "accuracy": acc,
            "dice": dsc,
        })

    mean_accuracy = float(np.mean([r["accuracy"] for r in case_results]))
    mean_dice = float(np.nanmean([r["dice"] for r in case_results]))
    final_score = 0.7 * mean_accuracy + 0.3 * mean_dice

    return {
        "mean_accuracy": mean_accuracy,
        "mean_dice": mean_dice,
        "final_score": final_score,
        "case_results": case_results,
    }

results = evaluate_detection_and_segmentation(DIR_GT_MASK, DIR_PRED_MASK)

print(f"Mean Accuracy : {results['mean_accuracy']:.4f}")
print(f"Mean Dice     : {results['mean_dice']:.4f}")
print(f"Final Score   : {results['final_score']:.4f}")